# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.01 · Etiquetado Ollama local o Hugging Face en Colab

Usa Ollama HTTP en local o, opcionalmente, Hugging Face sobre Colab L4; ambos conservan el mismo contrato y reanudan por `chunk_id`.

Ollama admite un JSON Schema en la salida estructurada y su validación posterior con Pydantic [1]. El backend local usa exactamente `qwen3.5:4b` y debe registrar el digest publicado por Ollama [2]; el backend Colab usa `Qwen/Qwen3-4B`, cuyo linaje se describe en el informe Qwen3 [3] y cuya revisión exacta consta en la tarjeta del modelo [4]. Las salidas LLM son propuestas de anotación y no *ground truth*: la anotación asistida en tareas subjetivas requiere controles humanos y de anclaje [5]. El prompt, el reintento y la precedencia son decisiones locales.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab L4 desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab` y asigne una **NVIDIA L4**. El notebook permanece local; Drive transporta solo el bundle mínimo verificado. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [6]. La integridad del bundle se comprueba con SHA-256 [7]. No sincronice cachés de modelos ni escriba checkpoints directamente en Drive.

In [ ]:
# Backend reproducible: local o Google Colab desde VS Code
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile

COLAB_NOTEBOOK_ID = "02_01"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = True
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            "Falta bundle_manifest.json. Prepare y suba resultados/colab_bundle a " + str(BUNDLE_DIR)
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    os.environ["MODPERU_ROOT"] = str(ROOT)
    os.environ["HF_HOME"] = "/content/huggingface"
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Selección del proveedor

In [ ]:
if COLAB_CONTEXT is not None:
    from moderacion_peru.providers import HuggingFaceProvider
    provider=HuggingFaceProvider(model='Qwen/Qwen3-4B',revision='1cfa9a7208912126459214e8b04321603b3df60c',device='cuda',max_new_tokens=512)
    SOURCE=COLAB_CONTEXT.input('chunks_v2')
    OUTPUT=COLAB_CONTEXT.scratch_output_dir/'huggingface_qwen3_4b_v2.jsonl'
    ERRORS=COLAB_CONTEXT.scratch_output_dir/'huggingface_qwen3_4b_v2.errors.jsonl'
else:
    from moderacion_peru.providers import OllamaProvider
    provider=OllamaProvider(model='qwen3.5:4b')
    SOURCE=ROOT/'datos/processed/chunks_v2.jsonl'
    OUTPUT=ROOT/'datos/etiquetado/local/ollama_qwen35_4b_v2.jsonl'
    ERRORS=ROOT/'datos/etiquetado/local/ollama_qwen35_4b_v2.errors.jsonl'
show_result('Estado del proveedor', provider.probe(), tone='success')
show_summary('Rutas de la campaña', {'entrada': SOURCE, 'salida': OUTPUT, 'errores': ERRORS}, tone='neutral')

## Etiquetado incremental

In [ ]:
from moderacion_peru.io import read_jsonl
from moderacion_peru.labeling import annotate_incremental
RUN=False
LIMIT=20  # Quite el límite solo después del smoke test
if RUN:
    show_result('Resultado del etiquetado incremental', annotate_incremental(read_jsonl(SOURCE),provider,OUTPUT,error_path=ERRORS,limit=LIMIT), tone='success')
else:
    show_callout('Preflight completo', 'Cambie RUN=True. La salida reanuda por chunk_id.', tone='neutral')

## Publicación o checkpoint en Drive

Los archivos se generan en el SSD efímero de `/content`. Active esta celda después de un checkpoint coherente o al finalizar; publica un solo TAR.GZ y luego su manifiesto.

In [ ]:
PUBLISH_TO_DRIVE = False
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación desactivada', 'Cambie PUBLISH_TO_DRIVE=True tras guardar un checkpoint consistente.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] Ollama, "Structured Outputs," Ollama Documentation, 2026. [Online]. Available: https://docs.ollama.com/capabilities/structured-outputs. Accessed: Aug. 5, 2026.

[2] Ollama, "Model Card: qwen3.5:4b," Ollama Model Library, 2026, model digest 2a654d98e6fb. [Online]. Available: https://ollama.com/library/qwen3.5:4b. Accessed: Aug. 5, 2026.

[3] A. Yang, A. Li, B. Yang, et al., "Qwen3 Technical Report," arXiv:2505.09388, 2025, doi: 10.48550/arXiv.2505.09388.

[4] Qwen Team, "Model Card: Qwen/Qwen3-4B," Hugging Face Hub, revision 1cfa9a7208912126459214e8b04321603b3df60c, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-4B/tree/1cfa9a7208912126459214e8b04321603b3df60c. Accessed: Aug. 5, 2026.

[5] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.

[6] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[7] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.